# 4.6 - Logistics System Size Diagnostics

Obiettivo: produrre metriche sintetiche per capire la grandezza del sistema logistico LPG.

Output principali:
- distanza media reseller -> filling di riferimento (in km, distanza geometrica),
- tempo medio reseller -> filling di riferimento (min),
- clienti medi serviti da un reseller,
- distribuzioni a decili (10%, 20%, ..., 100%) per reseller e filling,
- conteggi totali di reseller e filling e copertura delle assegnazioni.

Notebook solo di analisi: non modifica i file sorgente.

In [1]:
from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
CHAIN_GPKG = DATA_DIR / "chain_with_cost.gpkg"
SELL_POINT_GPKG = DATA_DIR / "sell_point_clients.gpkg"
FILLING_POINT_GPKG = DATA_DIR / "filling_point_clients.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
SELL_POINT_LAYER = "sell_point_clients"
FILLING_POINT_LAYER = "filling_point_clients"

ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
CLIENTS_COL = "clients"

for p in [CHAIN_GPKG, SELL_POINT_GPKG]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

print("Inputs found:")
print(f"- {CHAIN_GPKG}")
print(f"- {SELL_POINT_GPKG}")
print(f"- optional: {FILLING_POINT_GPKG} (exists={FILLING_POINT_GPKG.exists()})")

Inputs found:
- dataset_big\chain_with_cost.gpkg
- dataset_big\sell_point_clients.gpkg
- optional: dataset_big\filling_point_clients.gpkg (exists=True)


In [2]:
def decile_summary(series: pd.Series, name: str) -> pd.DataFrame:
    s = pd.to_numeric(series, errors="coerce")
    s = s[np.isfinite(s)]
    if len(s) == 0:
        return pd.DataFrame({"metric": [name], "note": ["no valid data"]})

    q = [i / 10 for i in range(1, 11)]
    vals = s.quantile(q).to_numpy(dtype=float)
    return pd.DataFrame({
        "metric": name,
        "percentile": [int(qq * 100) for qq in q],
        "value": vals,
    })


# 1) Load base layers from 4.4
resell = gpd.read_file(CHAIN_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(CHAIN_GPKG, layer=FILLING_LAYER)

if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")

for c in [ID_COL, FILL_REF_COL, FILL_TIME_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing column '{c}' in resell layer")
if ID_COL not in filling.columns:
    raise KeyError(f"Missing column '{ID_COL}' in filling layer")

# 2) Numeric cleanup
resell = resell.copy()
filling = filling.copy()
resell[ID_COL] = pd.to_numeric(resell[ID_COL], errors="coerce")
resell[FILL_REF_COL] = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
resell[FILL_TIME_COL] = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
filling[ID_COL] = pd.to_numeric(filling[ID_COL], errors="coerce")

resell = resell[resell[ID_COL].notna() & (resell[ID_COL] > 0)].copy()
filling = filling[filling[ID_COL].notna() & (filling[ID_COL] > 0)].copy()
resell[ID_COL] = resell[ID_COL].astype("int64")
filling[ID_COL] = filling[ID_COL].astype("int64")

# 3) Attach reseller clients from 4.3 output
sell_points = gpd.read_file(SELL_POINT_GPKG, layer=SELL_POINT_LAYER)
if ID_COL not in sell_points.columns or CLIENTS_COL not in sell_points.columns:
    raise KeyError(f"Missing required columns in {SELL_POINT_GPKG}: {ID_COL}, {CLIENTS_COL}")

sp = sell_points[[ID_COL, CLIENTS_COL]].copy()
sp[ID_COL] = pd.to_numeric(sp[ID_COL], errors="coerce")
sp[CLIENTS_COL] = pd.to_numeric(sp[CLIENTS_COL], errors="coerce")
sp = sp[sp[ID_COL].notna() & (sp[ID_COL] > 0)].copy()
sp[ID_COL] = sp[ID_COL].astype("int64")
sp = sp.drop_duplicates(subset=[ID_COL], keep="first")

resell = resell.merge(sp, on=ID_COL, how="left")
resell[CLIENTS_COL] = pd.to_numeric(resell[CLIENTS_COL], errors="coerce")

# 4) Merge assigned filling geometry and compute straight-line distances
fill_geom = filling[[ID_COL, "geometry"]].rename(columns={ID_COL: FILL_REF_COL, "geometry": "geometry_fill"})
resell_geo = resell[[ID_COL, FILL_REF_COL, FILL_TIME_COL, CLIENTS_COL, "geometry"]].copy()
resell_geo = resell_geo.rename(columns={"geometry": "geometry_res"})

merged = resell_geo.merge(fill_geom, on=FILL_REF_COL, how="left")
geom_ok = merged["geometry_res"].notna() & merged["geometry_fill"].notna()
merged["distance_km"] = np.nan
merged.loc[geom_ok, "distance_km"] = (
    merged.loc[geom_ok, "geometry_res"].distance(merged.loc[geom_ok, "geometry_fill"]).astype(float) / 1000.0
)

# 5) Assignment and aggregate filling metrics
valid_ref = merged[FILL_REF_COL].notna() & (merged[FILL_REF_COL] > 0)
valid_time = np.isfinite(pd.to_numeric(merged[FILL_TIME_COL], errors="coerce")) & (merged[FILL_TIME_COL] >= 0) & (merged[FILL_TIME_COL] < 7000)
valid_assignment = valid_ref & valid_time & merged["geometry_fill"].notna()

assigned = merged.loc[valid_assignment, [FILL_REF_COL, CLIENTS_COL, "distance_km"]].copy()
assigned[CLIENTS_COL] = pd.to_numeric(assigned[CLIENTS_COL], errors="coerce")

filling_sizes = (
    assigned.groupby(FILL_REF_COL, as_index=False)
    .agg(
        assigned_resellers=(FILL_REF_COL, "size"),
        assigned_clients_sum=(CLIENTS_COL, "sum"),
        assigned_clients_mean=(CLIENTS_COL, "mean"),
        assigned_dist_km_mean=("distance_km", "mean"),
    )
)

# Optional: integrate totals from filling_point_clients if available
if FILLING_POINT_GPKG.exists():
    try:
        fpc = gpd.read_file(FILLING_POINT_GPKG, layer=FILLING_POINT_LAYER)
        if ID_COL in fpc.columns:
            fpc[ID_COL] = pd.to_numeric(fpc[ID_COL], errors="coerce")
            fpc = fpc[fpc[ID_COL].notna() & (fpc[ID_COL] > 0)].copy()
            fpc[ID_COL] = fpc[ID_COL].astype("int64")
            keep = [ID_COL]
            for c in ["total_fil_clients", "total_max_ideal_clients"]:
                if c in fpc.columns:
                    keep.append(c)
            filling_sizes = filling_sizes.merge(
                fpc[keep].rename(columns={ID_COL: FILL_REF_COL}),
                on=FILL_REF_COL,
                how="left",
            )
    except Exception as e:
        print(f"Warning: could not read optional filling_point_clients.gpkg ({e})")

# 6) Global KPIs
kpis = {
    "n_resellers_total": int(len(resell)),
    "n_fillings_total": int(len(filling)),
    "n_resellers_valid_assignment": int(valid_assignment.sum()),
    "assignment_rate": float(valid_assignment.mean()) if len(resell) else np.nan,
    "mean_distance_km": float(np.nanmean(merged.loc[valid_assignment, "distance_km"])) if valid_assignment.any() else np.nan,
    "median_distance_km": float(np.nanmedian(merged.loc[valid_assignment, "distance_km"])) if valid_assignment.any() else np.nan,
    "mean_time_min": float(np.nanmean(merged.loc[valid_assignment, FILL_TIME_COL])) if valid_assignment.any() else np.nan,
    "median_time_min": float(np.nanmedian(merged.loc[valid_assignment, FILL_TIME_COL])) if valid_assignment.any() else np.nan,
    "mean_clients_per_reseller": float(np.nanmean(merged[CLIENTS_COL])) if np.isfinite(merged[CLIENTS_COL]).any() else np.nan,
    "median_clients_per_reseller": float(np.nanmedian(merged[CLIENTS_COL])) if np.isfinite(merged[CLIENTS_COL]).any() else np.nan,
}

kpi_df = pd.DataFrame([kpis]).T.reset_index()
kpi_df.columns = ["metric", "value"]

print("=== KPI DI SISTEMA ===")
display(kpi_df)

print("\n=== ESEMPI PRIME RIGHE: reseller con clienti/distanze ===")
display(merged[[ID_COL, FILL_REF_COL, FILL_TIME_COL, CLIENTS_COL, "distance_km"]].head(10))

print("\n=== ESEMPI PRIME RIGHE: dimensione filling ===")
display(filling_sizes.head(10))

=== KPI DI SISTEMA ===


,metric,value
0,n_resellers_total,2413.000000
1,n_fillings_total,375.000000
2,n_resellers_valid_assignment,2413.000000
3,assignment_rate,1.000000
4,mean_distance_km,16.949165
5,median_distance_km,3.855228
6,mean_time_min,19.284126
7,median_time_min,4.000000
8,mean_clients_per_reseller,9364.140042
9,median_clients_per_reseller,5420.294384



=== ESEMPI PRIME RIGHE: reseller con clienti/distanze ===


,id_res&fil,filling_reference,filling_ref_time_min,clients,distance_km
0,1,2517,2.957143,10050.711541,2.193526
1,2,2572,10.400000,28016.575024,7.088660
2,3,2571,1.178571,1571.739478,0.750787
3,4,2576,5.185714,19757.141324,5.509917
4,5,2787,223.792862,16966.201963,256.815990
5,6,2505,86.228577,7126.323266,53.742083
6,7,2507,3.600000,15052.299726,1.612838
7,8,2533,10.428572,4805.450375,8.112418
8,9,2553,5.100000,30578.579505,2.806293
9,10,2572,17.200001,11991.894126,11.430281



=== ESEMPI PRIME RIGHE: dimensione filling ===


,filling_reference,assigned_resellers,assigned_clients_sum,assigned_clients_mean,assigned_dist_km_mean,total_fil_clients,total_max_ideal_clients
0,2415,7,92205.348904,13172.192701,7.444971,98672.456482,1.143915e+06
1,2416,3,11376.810382,3792.270127,0.722128,11376.810382,2.150842e+04
2,2417,8,82814.473566,10351.809196,5.647310,86057.592330,1.760282e+06
3,2418,4,84950.969433,21237.742358,1.795032,93115.962997,8.651414e+05
4,2419,6,15697.055455,2616.175909,1.223131,15697.055455,3.793334e+04
5,2420,11,55856.270454,5077.842769,2.120959,70020.469045,1.427260e+05
6,2421,2,18515.324433,9257.662216,6.489434,23100.922074,1.606487e+05
7,2422,4,14146.316977,3536.579244,5.189580,18397.157733,4.301410e+05
8,2423,5,40718.269598,8143.653920,14.104019,41300.100712,6.611525e+05
9,2424,7,52892.763332,7556.109047,3.133123,59389.529747,5.747848e+05


In [3]:
# Decili 10%-100% per capire la dimensione di reseller e filling
res_clients_dec = decile_summary(merged[CLIENTS_COL], "reseller_clients")
res_dist_dec = decile_summary(merged.loc[valid_assignment, "distance_km"], "reseller_distance_km")
res_time_dec = decile_summary(merged.loc[valid_assignment, FILL_TIME_COL], "reseller_time_min")

fill_reseller_count_dec = decile_summary(filling_sizes["assigned_resellers"], "filling_assigned_resellers")
fill_clients_sum_dec = decile_summary(filling_sizes["assigned_clients_sum"], "filling_assigned_clients_sum")

tables = [
    res_clients_dec,
    res_dist_dec,
    res_time_dec,
    fill_reseller_count_dec,
    fill_clients_sum_dec,
]

print("=== DECILI (10%-100%) ===")
for t in tables:
    print()
    if "percentile" in t.columns:
        print(f"Metric: {t['metric'].iloc[0]}")
        display(t)
    else:
        display(t)

=== DECILI (10%-100%) ===

Metric: reseller_clients


,metric,percentile,value
0,reseller_clients,10,0.000000
1,reseller_clients,20,0.000000
2,reseller_clients,30,2243.321171
3,reseller_clients,40,3780.228127
4,reseller_clients,50,5420.294384
5,reseller_clients,60,7577.536395
6,reseller_clients,70,10023.711015
7,reseller_clients,80,13782.069272
8,reseller_clients,90,20721.038230
9,reseller_clients,100,231629.163725



Metric: reseller_distance_km


,metric,percentile,value
0,reseller_distance_km,10,0.856616
1,reseller_distance_km,20,1.522048
2,reseller_distance_km,30,2.160288
3,reseller_distance_km,40,2.874072
4,reseller_distance_km,50,3.855228
5,reseller_distance_km,60,5.589584
6,reseller_distance_km,70,9.942148
7,reseller_distance_km,80,26.567457
8,reseller_distance_km,90,53.285672
9,reseller_distance_km,100,262.739039



Metric: reseller_time_min


,metric,percentile,value
0,reseller_time_min,10,0.857143
1,reseller_time_min,20,1.500000
2,reseller_time_min,30,2.082857
3,reseller_time_min,40,2.950000
4,reseller_time_min,50,4.000000
5,reseller_time_min,60,5.744286
6,reseller_time_min,70,11.000000
7,reseller_time_min,80,30.320000
8,reseller_time_min,90,61.900002
9,reseller_time_min,100,1087.938110



Metric: filling_assigned_resellers


,metric,percentile,value
0,filling_assigned_resellers,10,1.0
1,filling_assigned_resellers,20,2.0
2,filling_assigned_resellers,30,2.0
3,filling_assigned_resellers,40,3.0
4,filling_assigned_resellers,50,4.0
5,filling_assigned_resellers,60,6.0
6,filling_assigned_resellers,70,8.0
7,filling_assigned_resellers,80,12.6
8,filling_assigned_resellers,90,18.8
9,filling_assigned_resellers,100,59.0



Metric: filling_assigned_clients_sum


,metric,percentile,value
0,filling_assigned_clients_sum,10,8126.988140
1,filling_assigned_clients_sum,20,14866.559336
2,filling_assigned_clients_sum,30,22523.720826
3,filling_assigned_clients_sum,40,32029.055594
4,filling_assigned_clients_sum,50,40718.269598
5,filling_assigned_clients_sum,60,57449.758306
6,filling_assigned_clients_sum,70,87006.625396
7,filling_assigned_clients_sum,80,119247.676964
8,filling_assigned_clients_sum,90,165233.716820
9,filling_assigned_clients_sum,100,789602.714477


In [4]:
# Classi per decile (conteggio osservazioni in ogni 10%)
def decile_bins_count(series: pd.Series, label: str) -> pd.DataFrame:
    s = pd.to_numeric(series, errors="coerce")
    s = s[np.isfinite(s)]
    if len(s) < 10:
        return pd.DataFrame({"metric": [label], "note": ["not enough valid rows for 10 bins"]})

    # duplicates='drop' evita errori quando molti valori sono uguali
    cats = pd.qcut(s, q=10, duplicates="drop")
    out = cats.value_counts(sort=False).reset_index()
    out.columns = ["bin", "count"]
    out["share"] = out["count"] / out["count"].sum()
    out.insert(0, "metric", label)
    return out

bin_tables = [
    decile_bins_count(merged[CLIENTS_COL], "reseller_clients"),
    decile_bins_count(merged.loc[valid_assignment, "distance_km"], "reseller_distance_km"),
    decile_bins_count(filling_sizes["assigned_resellers"], "filling_assigned_resellers"),
    decile_bins_count(filling_sizes["assigned_clients_sum"], "filling_assigned_clients_sum"),
]

print("=== CLASSI A DECILI (conteggi) ===")
for bt in bin_tables:
    print()
    display(bt)

=== CLASSI A DECILI (conteggi) ===



,metric,bin,count,share
0,reseller_clients,"(-0.001, 2243.321]",724,0.300041
1,reseller_clients,"(2243.321, 3780.228]",241,0.099876
2,reseller_clients,"(3780.228, 5420.294]",242,0.100290
3,reseller_clients,"(5420.294, 7577.536]",241,0.099876
4,reseller_clients,"(7577.536, 10023.711]",241,0.099876
5,reseller_clients,"(10023.711, 13782.069]",241,0.099876
6,reseller_clients,"(13782.069, 20721.038]",241,0.099876
7,reseller_clients,"(20721.038, 231629.164]",242,0.100290


,metric,bin,count,share
0,reseller_distance_km,"(-0.001, 0.857]",242,0.100290
1,reseller_distance_km,"(0.857, 1.522]",241,0.099876
2,reseller_distance_km,"(1.522, 2.16]",241,0.099876
3,reseller_distance_km,"(2.16, 2.874]",241,0.099876
4,reseller_distance_km,"(2.874, 3.855]",242,0.100290
5,reseller_distance_km,"(3.855, 5.59]",241,0.099876
6,reseller_distance_km,"(5.59, 9.942]",241,0.099876
7,reseller_distance_km,"(9.942, 26.567]",241,0.099876
8,reseller_distance_km,"(26.567, 53.286]",241,0.099876
9,reseller_distance_km,"(53.286, 262.739]",242,0.100290


,metric,bin,count,share
0,filling_assigned_resellers,"(0.999, 2.0]",101,0.322684
1,filling_assigned_resellers,"(2.0, 3.0]",33,0.105431
2,filling_assigned_resellers,"(3.0, 4.0]",29,0.092652
3,filling_assigned_resellers,"(4.0, 6.0]",36,0.115016
4,filling_assigned_resellers,"(6.0, 8.0]",22,0.070288
5,filling_assigned_resellers,"(8.0, 12.6]",29,0.092652
6,filling_assigned_resellers,"(12.6, 18.8]",31,0.099042
7,filling_assigned_resellers,"(18.8, 59.0]",32,0.102236


,metric,bin,count,share
0,filling_assigned_clients_sum,"(-0.001, 8126.988]",32,0.102236
1,filling_assigned_clients_sum,"(8126.988, 14866.559]",31,0.099042
2,filling_assigned_clients_sum,"(14866.559, 22523.721]",31,0.099042
3,filling_assigned_clients_sum,"(22523.721, 32029.056]",31,0.099042
4,filling_assigned_clients_sum,"(32029.056, 40718.27]",32,0.102236
5,filling_assigned_clients_sum,"(40718.27, 57449.758]",31,0.099042
6,filling_assigned_clients_sum,"(57449.758, 87006.625]",31,0.099042
7,filling_assigned_clients_sum,"(87006.625, 119247.677]",31,0.099042
8,filling_assigned_clients_sum,"(119247.677, 165233.717]",31,0.099042
9,filling_assigned_clients_sum,"(165233.717, 789602.714]",32,0.102236


In [6]:
"""
Export all diagnostics tables to CSV (single long-format file).

This cell expects previous cells already executed and exports:
- KPI table
- decile tables (10%-100%)
- decile-bin tables (count/share)

Output:
- dataset_big/logistics_diagnostics_all_tables.csv
"""

from pathlib import Path

import numpy as np
import pandas as pd

EXPORT_PATH = DATA_DIR / "logistics_diagnostics_all_tables.csv"

def _normalize_for_export(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    out = df.copy()

    # Convert interval/bin objects to string to avoid CSV serialization ambiguity.
    for col in out.columns:
        if str(out[col].dtype) == "category":
            out[col] = out[col].astype(str)
    if "bin" in out.columns:
        out["bin"] = out["bin"].astype(str)

    out.insert(0, "table_name", table_name)
    out.insert(1, "row_id", np.arange(1, len(out) + 1, dtype=int))
    return out

export_frames = []

# KPI
kpi_export = kpi_df.copy()
kpi_export = kpi_export.rename(columns={"metric": "name", "value": "value"})
export_frames.append(_normalize_for_export(kpi_export, "kpi"))

# Deciles
for i, t in enumerate(tables, start=1):
    metric_name = str(t["metric"].iloc[0]) if ("metric" in t.columns and len(t) > 0) else f"decile_{i}"
    export_frames.append(_normalize_for_export(t, f"deciles::{metric_name}"))

# Decile bins
for i, bt in enumerate(bin_tables, start=1):
    metric_name = str(bt["metric"].iloc[0]) if ("metric" in bt.columns and len(bt) > 0) else f"bins_{i}"
    export_frames.append(_normalize_for_export(bt, f"decile_bins::{metric_name}"))

# Optional: add first-level filling summary table as well
if "filling_sizes" in globals() and isinstance(filling_sizes, pd.DataFrame) and (len(filling_sizes) > 0):
    export_frames.append(_normalize_for_export(filling_sizes, "filling_sizes_summary"))

all_tables_export = pd.concat(export_frames, ignore_index=True, sort=False)

# Reorder a few commonly used columns near the front when present.
front_cols = ["table_name", "row_id", "metric", "name", "percentile", "value", "count", "share", "bin"]
existing_front = [c for c in front_cols if c in all_tables_export.columns]
other_cols = [c for c in all_tables_export.columns if c not in existing_front]
all_tables_export = all_tables_export[existing_front + other_cols]

all_tables_export.to_csv(EXPORT_PATH, index=False, encoding="utf-8")

print("CSV export completed.")
print(f"File: {EXPORT_PATH}")
print(f"Rows exported: {len(all_tables_export):,}")
print("Tables exported:")
for name in all_tables_export["table_name"].dropna().unique().tolist():
    print(f"- {name}")

CSV export completed.
File: dataset_big\logistics_diagnostics_all_tables.csv
Rows exported: 409
Tables exported:
- kpi
- deciles::reseller_clients
- deciles::reseller_distance_km
- deciles::reseller_time_min
- deciles::filling_assigned_resellers
- deciles::filling_assigned_clients_sum
- decile_bins::reseller_clients
- decile_bins::reseller_distance_km
- decile_bins::filling_assigned_resellers
- decile_bins::filling_assigned_clients_sum
- filling_sizes_summary
